# 06 — End-of-Day Survey + Clamping (Issue 8)

Demonstrates the end-of-day survey re-administration with opinion clamping:
1. Load data, create agents, run broadcast + peer phases (or mock reflections)
2. Administer end-of-day survey with reflection-aware context
3. Apply opinion shift clamping (max_shift parameter)
4. Analyze raw vs clamped opinion distributions

**Covers:** Issue 8 (End-of-Day Survey + Clamping)  
**Depends on:** Issues 3, 6, 7

In [ ]:
import os, sys, random
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

from cag.io.survey import load
from cag.abm.agent import SurveyedCitizen, PoliticalAgent
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS
from cag.io.llm import load_api_key

# Attribute maps and IDs
from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)
import pandas as pd
import matplotlib.pyplot as plt

print("Imports OK")

## 1. Load Data & Build Environment

In [ ]:
random.seed(42)
year = 2026

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData_train.csv")

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

sn.political_agent_a = PoliticalAgent("agent_a", "pro_climate")
sn.political_agent_b = PoliticalAgent("agent_b", "anti_climate")
sn.assign_political_exposure()
sn.create_network(seed=42)
sn.assign_network_blocks()

api_key = load_api_key("../data/api_key.csv")
target_policy = ClimatePolicyID.CARBON_TAX

print(f"Total citizens: {len(sn.agents_active)}")

## 2. Baseline Survey (Day 0)

Run baseline survey for a small demo subset. Day 0 has no clamping (raw == clamped).

In [ ]:
# TODO: Run baseline survey for demo subset
# MAX_DEMO = 10
# baseline_df = sn.run_baseline(api_key=api_key, model="gpt-4o-mini")
# baseline_df.head()

## 3. Simulate Day 1 Interactions (Mock or Real)

Run P-A, P-B, and C phases to accumulate reflections before the end-of-day survey.

In [ ]:
# TODO: Run broadcast + peer phases (or inject mock reflections for testing)

## 4. End-of-Day Survey (Day 1)

Survey context includes all reflections from Day 1. Previous response is referenced for anchoring.

In [ ]:
# TODO: Run end-of-day survey
# eod_df = sn.run_end_of_day_survey(
#     policy_id=target_policy, day=1, max_shift=1,
#     api_key=api_key, model="gpt-4o-mini",
# )
# eod_df.head()

## 5. Raw vs Clamped Opinion Distribution

Compare raw LLM responses to clamped values. How many agents were clamped?

In [ ]:
# TODO: Plot raw vs clamped distributions
# - Histogram of raw_numeric vs clamped_numeric
# - Count of agents where raw != clamped
# - Mean shift magnitude

## 6. Opinion History Inspection

Verify `opinion_history` stores `(day, raw, clamped)` tuples correctly.

In [ ]:
# TODO: Inspect opinion_history for a sample citizen
# - Day 0: raw == clamped (no clamping on baseline)
# - Day 1: raw may differ from clamped

## 7. Clamping Verification

In [ ]:
# TODO: Sanity checks
# - No clamped shift exceeds max_shift
# - Day 0 raw == clamped for all agents
# - opinion_history has correct tuple structure (day, raw, clamped)
# - DataFrame schema matches spec

## 8. LLM Call Summary

In [ ]:
# TODO: LLM call cost summary for end-of-day survey